In [91]:
import pandas as pd

from pypdf import PdfReader
from pathlib import Path
import pdfplumber

In [92]:
data_folder = Path("../data/raw")

csv_files = list(data_folder.glob("*.csv"))
pdf_files = list(data_folder.glob("*.pdf"))

print("CSV files:")
for file in csv_files:
    print(file)

print("\nPDF files:")
for file in pdf_files:
    print(file)

CSV files:
..\data\raw\2022-COMMUNITY-PROJECTS-PETAUKE-CENTRAL.csv
..\data\raw\2025-NOT-APPROVED-COMMUNITY-PROJECTS-KAUMBWE2.csv
..\data\raw\2025-PROPOSED-COMMUNITY-PROJECTS-KAUMBWE2.csv

PDF files:
..\data\raw\11th-July-2025-Council-Minutes.pdf
..\data\raw\30th-April-2025- Council-Minutes.pdf
..\data\raw\Petauke-Town-Council-2025-OBB-Final-05.12.2024_Signed.pdf
..\data\raw\Petauke-Town-Council-2026-Budget-2026.pdf
..\data\raw\Petauke-Town-Council-Stratplan_2019-23.pdf
..\data\raw\Petauke.Lusangazi-Joint-IDP-Final.pdf


In [93]:
csv_data = {}

for file in csv_files:
    df = pd.read_csv(file)
    csv_data[file.name] = df

print(csv_data.keys())

dict_keys(['2022-COMMUNITY-PROJECTS-PETAUKE-CENTRAL.csv', '2025-NOT-APPROVED-COMMUNITY-PROJECTS-KAUMBWE2.csv', '2025-PROPOSED-COMMUNITY-PROJECTS-KAUMBWE2.csv'])


In [94]:
pdf_data = {}

for file in pdf_files:
    reader = PdfReader(file)

    text = ""

    for page in reader.pages:
        text += page.extract_text() or ""

    pdf_data[file.name] = text

print(pdf_data.keys())

dict_keys(['11th-July-2025-Council-Minutes.pdf', '30th-April-2025- Council-Minutes.pdf', 'Petauke-Town-Council-2025-OBB-Final-05.12.2024_Signed.pdf', 'Petauke-Town-Council-2026-Budget-2026.pdf', 'Petauke-Town-Council-Stratplan_2019-23.pdf', 'Petauke.Lusangazi-Joint-IDP-Final.pdf'])




INSPECTING "2026 BUDGET" AND "2025 OBB FINAL"

In [95]:
data_folder = Path("../data/raw") 
output_folder = Path("../data/processed") 
output_folder.mkdir(parents=True, exist_ok=True)

In [96]:
for pdf_file in pdf_files:

    if "2025-OBB" in pdf_file.name or "2026-Budget" in pdf_file.name:

        print(f"\nProcessing: {pdf_file.name}")

        extracted_tables = []

        with pdfplumber.open(pdf_file) as pdf:

            for page_num, page in enumerate(pdf.pages, start=1):

                tables = page.extract_tables()

                for table in tables:

                    if table:

                        df = pd.DataFrame(table)

                        # Remove empty rows and columns
                        df = df.fillna("")
                        df = df.map(
                            lambda cell: cell.strip()
                            if isinstance(cell, str)
                            else cell
                        )

                        df = df.loc[~(df == "").all(axis=1)]
                        df = df.loc[:, ~(df == "").all(axis=0)]

                        if not df.empty:
                            df["source_page"] = page_num
                            extracted_tables.append(df)

        # Save extracted tables
        if extracted_tables:

            final_df = pd.concat(
                extracted_tables,
                ignore_index=True
            )

            clean_name = (
                pdf_file.stem
                .lower()
                .replace("-", "_")
                .replace(".", "_")
            )

            output_file = (
                output_folder /
                f"db-unza26-csc4792-{clean_name}.csv"
            )

            final_df.to_csv(
                output_file,
                sep="|",
                index=False
            )

            print(f"Saved: {output_file}")
            print(f"Rows: {len(final_df)}")
            print(f"Columns: {len(final_df.columns)}")

        else:
            print("No tables found.")


Processing: Petauke-Town-Council-2025-OBB-Final-05.12.2024_Signed.pdf
Saved: ..\data\processed\db-unza26-csc4792-petauke_town_council_2025_obb_final_05_12_2024_signed.csv
Rows: 422
Columns: 9

Processing: Petauke-Town-Council-2026-Budget-2026.pdf
Saved: ..\data\processed\db-unza26-csc4792-petauke_town_council_2026_budget_2026.csv
Rows: 446
Columns: 9


In [97]:
processed_files = list(output_folder.glob("*.csv"))

print("Processed CSV files:")

for file in processed_files:
    print(file)

Processed CSV files:
..\data\processed\db-unza26-csc4792-petauke_town_council_2025_obb_final_05_12_2024_signed.csv
..\data\processed\db-unza26-csc4792-petauke_town_council_2026_budget_2026.csv


In [98]:
obb_dataset = pd.read_csv(
    "../data/processed/db-unza26-csc4792-petauke_town_council_2025_obb_final_05_12_2024_signed.csv",
    sep="|"
)

obb_dataset.head()

,0,1,2,3,5,7,source_page,4,6
0,CODE,REVENUE DESCRIPTION,APPROVED\nBUDGET 2025,NaN,REVISED\nBUDGET 2026,BUDGET\nESTIMATE 2027,2,NaN,NaN
1,01,Local taxes/rates,NaN,NaN,NaN,NaN,2,NaN,NaN
2,001,Residential,"2,000,000",NaN,"2,000,000","2,000,000",2,NaN,NaN
3,002,Commercial,"2,000,000",NaN,"2,000,000","2,000,000",2,NaN,NaN
4,NaN,SubItem Total,NaN,"4,000,000","4,000,000","4,000,000",2,NaN,NaN


In [99]:
budget_dataset = pd.read_csv(
    "../data/processed/db-unza26-csc4792-petauke_town_council_2026_budget_2026.csv",
    sep="|"
)

budget_dataset


,0,1,2,3,5,7,source_page,4,6
0,CODE,REVENUE DESCRIPTION,APPROVED\nBUDGET 2026,NaN,REVISED\nBUDGET 2027,BUDGET\nESTIMATE 2028,2,NaN,NaN
1,01,Local taxes/rates,NaN,NaN,NaN,NaN,2,NaN,NaN
2,001,Residential,"2,000,000",NaN,"2,000,000","2,000,000",2,NaN,NaN
3,002,Commercial,"2,000,000",NaN,"2,000,000","2,000,000",2,NaN,NaN
4,NaN,SubItem Total,NaN,"4,000,000","4,000,000","4,000,000",2,NaN,NaN
...,...,...,...,...,...,...,...,...,...
441,Cash for Work Initiative Implemented\n01 Numbe...,-\n-,-\n-,"58,068\n100","23,616\n92",NaN,47,"69,330\n-",NaN
442,Functional Literacy literacy students enrolled...,-\n-\n-,-\n-\n-,100\n-\n-,80\n4\n4,NaN,47,-\n-\n-,NaN
443,Geographic\nLocation,Key Outputs and Outputs Indicator,MTEF Output Target,NaN,NaN,NaN,48,NaN,NaN
444,NaN,NaN,2022,2023,NaN,NaN,48,2024,NaN


In [100]:
obb_dataset.columns = obb_dataset.iloc[0]

obb_dataset = obb_dataset.iloc[1:].reset_index(drop=True)

In [101]:
obb_dataset = obb_dataset.fillna("")
obb_dataset

,CODE,REVENUE DESCRIPTION,APPROVED\nBUDGET 2025,NaN,REVISED\nBUDGET 2026,BUDGET\nESTIMATE 2027,2,NaN,NaN
0,01,Local taxes/rates,,,,,2,,
1,001,Residential,"2,000,000",,"2,000,000","2,000,000",2,,
2,002,Commercial,"2,000,000",,"2,000,000","2,000,000",2,,
3,,SubItem Total,,"4,000,000","4,000,000","4,000,000",2,,
4,001,Personal levy,"73,500",,"80,850","97,020",2,,
...,...,...,...,...,...,...,...,...,...
416,Older persons cared for\n01 Number of older pe...,-,-,-,50,,47,-,
417,Cash for Work Initiative implemented\n01 Numbe...,-\n-,-\n-,-\n-,"58,068\n100",,47,-\n-,
418,Poor and vulnerable learners trained under fun...,-,-,-,100,,47,-,
419,Geographic\nLocation,Key Outputs and Outputs Indicator,MTEF Output Target,,,,48,,


In [119]:
obb_dataset.columns

Index(['code', 'description', 'approved_budget', 'actual_expenditure',
       'budget_estimate_2026', 'budget_estimate_2028', 'source_page',
       'revised_budget', 'budget_estimate_2027'],
      dtype='str')

In [120]:
# Standard Pandas syntax
obb_dataset = obb_dataset.drop(
    columns=["source_page", "revised_budget", "budget_estimate_2027"]
)

In [121]:
obb_dataset.columns

Index(['code', 'description', 'approved_budget', 'actual_expenditure',
       'budget_estimate_2026', 'budget_estimate_2028'],
      dtype='str')

In [122]:
obb_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 385 entries, 0 to 384
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   code                  385 non-null    str  
 1   description           385 non-null    str  
 2   approved_budget       385 non-null    str  
 3   actual_expenditure    385 non-null    str  
 4   budget_estimate_2026  385 non-null    str  
 5   budget_estimate_2028  385 non-null    str  
dtypes: str(6)
memory usage: 18.2 KB


In [123]:

obb_dataset.isnull().sum()


code                    0
description             0
approved_budget         0
actual_expenditure      0
budget_estimate_2026    0
budget_estimate_2028    0
dtype: int64

In [124]:
obb_dataset

,code,description,approved_budget,actual_expenditure,budget_estimate_2026,budget_estimate_2028
0,01,Local taxes/rates,,,,
1,001,Residential,"2,000,000",,"2,000,000","2,000,000"
2,002,Commercial,"2,000,000",,"2,000,000","2,000,000"
3,,SubItem Total,,"4,000,000","4,000,000","4,000,000"
4,001,Personal levy,"73,500",,"80,850","97,020"
...,...,...,...,...,...,...
380,Cash for Work Initiative Implemented 01 Number...,- -,- -,"58,068 100","23,616 92",
381,Functional Literacy literacy students enrolled...,- - -,- - -,100 - -,80 4 4,
382,Geographic Location,Key Outputs and Outputs Indicator,MTEF Output Target,,,
383,,,2022,2023,,


In [125]:
obb_dataset.duplicated().sum()

np.int64(98)

In [126]:
obb_dataset = obb_dataset.drop_duplicates()

In [127]:
obb_dataset

,code,description,approved_budget,actual_expenditure,budget_estimate_2026,budget_estimate_2028
0,01,Local taxes/rates,,,,
1,001,Residential,"2,000,000",,"2,000,000","2,000,000"
2,002,Commercial,"2,000,000",,"2,000,000","2,000,000"
3,,SubItem Total,,"4,000,000","4,000,000","4,000,000"
4,001,Personal levy,"73,500",,"80,850","97,020"
...,...,...,...,...,...,...
380,Cash for Work Initiative Implemented 01 Number...,- -,- -,"58,068 100","23,616 92",
381,Functional Literacy literacy students enrolled...,- - -,- - -,100 - -,80 4 4,
382,Geographic Location,Key Outputs and Outputs Indicator,MTEF Output Target,,,
383,,,2022,2023,,


In [116]:
obb_dataset

,code,description,approved_budget,actual_expenditure,budget_estimate_2026,budget_estimate_2028,source_page,revised_budget,budget_estimate_2027
0,01,Local taxes/rates,,,,,2,,
1,001,Residential,"2,000,000",,"2,000,000","2,000,000",2,,
2,002,Commercial,"2,000,000",,"2,000,000","2,000,000",2,,
3,,SubItem Total,,"4,000,000","4,000,000","4,000,000",2,,
4,001,Personal levy,"73,500",,"80,850","97,020",2,,
...,...,...,...,...,...,...,...,...,...
380,Cash for Work Initiative Implemented 01 Number...,- -,- -,"58,068 100","23,616 92",,47,"69,330 -",
381,Functional Literacy literacy students enrolled...,- - -,- - -,100 - -,80 4 4,,47,- - -,
382,Geographic Location,Key Outputs and Outputs Indicator,MTEF Output Target,,,,48,,
383,,,2022,2023,,,48,2024,


In [128]:
obb_clean_dataset = obb_dataset.copy()

In [129]:
obb_clean_dataset = obb_clean_dataset[
    ~obb_clean_dataset["description"].str.contains(
        "SubItem Total|Key Outputs|Geographic Location|Animal Health Extension",
        case=False,
        na=False,
    )
]
obb_clean_dataset = obb_clean_dataset[
    ~obb_clean_dataset["code"].str.contains(
        "Cash for Work|Functional Literacy", case=False, na=False
    )
]

In [130]:
budget_cols = [
    "approved_budget",
    "actual_expenditure",
    "budget_estimate_2026",
    "budget_estimate_2028",
]

for col in budget_cols:
    # Replace dashes, empty strings, and spaces with NaN
    obb_clean_dataset[col] = (
        obb_clean_dataset[col]
        .astype(str)
        .str.replace(",", "")
        .str.replace(r"^\s*-\s*$", "", regex=True)
        .str.strip()
    )
    obb_clean_dataset[col] = pd.to_numeric(obb_clean_dataset[col], errors="coerce")

# 3. Clean Budget Codes
obb_clean_dataset["code"] = obb_clean_dataset["code"].astype(str).str.strip()

# Drop rows where description or code is entirely empty
obb_clean_dataset = obb_clean_dataset.dropna(subset=["description"]).reset_index(drop=True)

In [131]:
obb_clean_dataset

,code,description,approved_budget,actual_expenditure,budget_estimate_2026,budget_estimate_2028
0,01,Local taxes/rates,NaN,NaN,NaN,NaN
1,001,Residential,2000000.0,NaN,2000000.0,2000000.0
2,002,Commercial,2000000.0,NaN,2000000.0,2000000.0
3,001,Personal levy,73500.0,NaN,80850.0,97020.0
4,02,Fees and Charges,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
267,Programme Total,-,NaN,3683908.0,NaN,NaN
268,Welfare and Counselling Services Provided 01 N...,-,NaN,10.0,10.0,NaN
269,Welfare Services to GBVs Survivors Provided 01...,-,NaN,20.0,20.0,NaN
270,Older Persons Cared 01 Number of older persons...,-,NaN,50.0,50.0,NaN


In [132]:
obb_clean_dataset = obb_dataset.fillna("")
obb_clean_dataset

,code,description,approved_budget,actual_expenditure,budget_estimate_2026,budget_estimate_2028
0,01,Local taxes/rates,,,,
1,001,Residential,"2,000,000",,"2,000,000","2,000,000"
2,002,Commercial,"2,000,000",,"2,000,000","2,000,000"
3,,SubItem Total,,"4,000,000","4,000,000","4,000,000"
4,001,Personal levy,"73,500",,"80,850","97,020"
...,...,...,...,...,...,...
380,Cash for Work Initiative Implemented 01 Number...,- -,- -,"58,068 100","23,616 92",
381,Functional Literacy literacy students enrolled...,- - -,- - -,100 - -,80 4 4,
382,Geographic Location,Key Outputs and Outputs Indicator,MTEF Output Target,,,
383,,,2022,2023,,


In [ ]:
# Save as standard comma-separated CSV
obb_dataset.to_csv("obb_dataset_final.csv", index=False)